# The Hitchhiker’s Guide to Feature Extraction

[Reference](https://www.kdnuggets.com/2019/06/hitchhikers-guide-feature-extraction.html)

Ignore the warning about importing a deprecated moudule.

First, we load the data as explained in [feature tools documentation](https://featuretools.alteryx.com/en/stable/)

In [ ]:
import featuretools as ft

data = ft.demo.load_mock_customer()

In [ ]:
customers_df = data["customers"]
customers_df

In [ ]:
sessions_df = data["sessions"]
sessions_df.sample(5)

In [ ]:
transactions_df = data["transactions"]
transactions_df.sample(5)

Then we can continue reading the article.

## 1. Automatic Feature Creation using featuretools:

![Data relationships](./images/reference/featuretools_data_relationships.png)

In [ ]:
customers_df.head()

In [ ]:
sessions_df.head()

In [ ]:
transactions_df.head()

In [ ]:
# Create new entityset
es = ft.EntitySet(id = 'customers')

In [ ]:
# Create an entity from the customers dataframe

'''
DEPRECATED
es = es.entity_from_dataframe(entity_id = 'customers', dataframe = customers_df, index = 'customer_id', time_index = 'join_date' ,variable_types = {"zip_code": ft.variable_types.ZIPCode})
'''
from woodwork.logical_types import PostalCode

# new way to create an entity from the customers dataframe
es = es.add_dataframe(
    dataframe_name="customers",
    dataframe=customers_df,
    index="customer_id",
    time_index="join_date",
    logical_types={"zip_code": PostalCode},
)

In [ ]:
es

In [ ]:
feature_matrix, feature_defs = ft.dfs(
        entityset=es,
        target_dataframe_name="customers",
        max_depth=2,
    )

print(f"So far we have {len(feature_defs)} features in the entity set.")


In [ ]:
'''
DEPRECATED
# adding the transactions_df
es = es.entity_from_dataframe(entity_id="transactions",
                                 dataframe=transactions_df,
                                 index="transaction_id",
                               time_index="transaction_time",
                               variable_types={"product_id": ft.variable_types.Categorical})

# adding sessions_df
es = es.entity_from_dataframe(entity_id="sessions",
            dataframe=sessions_df,
            index="session_id", time_index = 'session_start')
'''

from woodwork.logical_types import Categorical

# Add transactions
es = es.add_dataframe(
    dataframe_name="transactions",
    dataframe=transactions_df,
    index="transaction_id",
    time_index="transaction_time",
    logical_types={"product_id": Categorical},
)

# Add sessions
es = es.add_dataframe(
    dataframe_name="sessions",
    dataframe=sessions_df,
    index="session_id",
    time_index="session_start",
    logical_types={"device": Categorical}, # This prevents the warning 'UserWarning: Could not infer format, so each element will be parsed individually, falling back to dateutil.'
)

In [ ]:
es

In [ ]:
'''
DEPRECATED
# adding the customer_id relationship
cust_relationship = ft.Relationship(es["customers"]["customer_id"],
                       es["sessions"]["customer_id"])

# Add the relationship to the entity set
es = es.add_relationship(cust_relationship)

# adding the session_id relationship
sess_relationship = ft.Relationship(es["sessions"]["session_id"],
                       es["transactions"]["session_id"])

# Add the relationship to the entity set
es = es.add_relationship(sess_relationship)
'''

# One customer → many sessions
es = es.add_relationship(
    parent_dataframe_name="customers",
    parent_column_name="customer_id",
    child_dataframe_name="sessions",
    child_column_name="customer_id",
)

# One session → many transactions
es = es.add_relationship(
    parent_dataframe_name="sessions",
    parent_column_name="session_id",
    child_dataframe_name="transactions",
    child_column_name="session_id",
)

es

In [ ]:
# DEPRECATED: feature_matrix, feature_defs = ft.dfs(entityset=es, target_entity="customers",max_depth = 2)

import warnings

with warnings.catch_warnings():
    # Suppress the FutureWarning about SeriesGroupBy in featuretools
    warnings.filterwarnings(
        "ignore",
        message=r"The provided callable .* is currently using SeriesGroupBy\.",
        category=FutureWarning,
    )
    feature_matrix, feature_defs = ft.dfs(
        entityset=es,
        target_dataframe_name="customers",
        max_depth=2,
    )

feature_matrix.head()

In [ ]:
feature_matrix.columns.tolist()

In [ ]:
print(f"Now we have {len(feature_defs)} features in the entity set.")

In the original [tutorial](https://www.kdnuggets.com/2019/06/hitchhikers-guide-feature-extraction.html) generates 73 features, whereas this notebook generates 75. This difference may be due to changes in Featuretools since the tutorial was published in 2019. This notebook uses Featuretools 1.31.0.

## 2. Handling Categorical Features: Label/Binary/Hashing and Target/Mean Encoding

### One hot encoding

In [ ]:
sessions_df["device"].value_counts()

In [ ]:
import pandas as pd
pd.get_dummies(sessions_df['device'],drop_first=True)


### OrdinalEncoding

Sometimes there is an order associated with categories. In such a case, I usually use a simple map/apply function in pandas to create a new ordinal column.

For example, if I had a dataframe containing temperature as three levels: high medium and low, I would encode that as:

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "City": ["London", "New York", "Dubai"],
    "Temperature": ["low", "medium", "high"],
})


In [ ]:
map_dict = {'low':0,'medium':1,'high':2}
def map_values(x):
    return map_dict[x]
df['Temperature_oe'] = df['Temperature'].apply(lambda x: map_values(x))

df

### LabelEncoder

We could also have used LabelEncoder to encode our variable to numbers. What a label encoder essentially does is that it sees the first value in the column and converts it to 0, next value to 1 and so on. This approach works reasonably well with tree models, and I end up using it when I have a lot of levels in the categorical variable. We can use this as:



In [ ]:
from sklearn.preprocessing import LabelEncoder
# create a labelencoder object
le = LabelEncoder()
# fit and transform on the data
sessions_df['device_le'] = le.fit_transform(sessions_df['device'])
sessions_df.head()

### BinaryEncoder

BinaryEncoder is another method that one can use to encode categorical variables. It is an excellent method to use if you have many levels in a column. While we can encode a column with 1024 levels using 1023 columns using One Hot Encoding, using Binary encoding we can do it by just using ten columns.

Let us say we have a column in our FIFA 19 player data that contains all club names. This column has 652 unique values. One Hot encoding means creating 651 columns that would mean a lot of memory usage and a lot of sparse columns.

If we use Binary encoder, we will only need ten columns as 2⁹<652 <2¹⁰.

We can binaryEncode this variable easily by using BinaryEncoder object from category_encoders:

In [ ]:
import pandas as pd

players = pd.DataFrame({
    "Player": [
        "Alex Smith", "Leo Martin", "Omar Hassan", "Lucas Silva",
        "Noah Brown", "Hugo Garcia", "Adam Wilson", "Marco Rossi",
    ],
    "Club": [
        "Arsenal", "Barcelona", "Chelsea", "Juventus",
        "Arsenal", "Real Madrid", "Chelsea", "Barcelona",
    ],
    "Age": [22, 25, 28, 24, 21, 30, 26, 23],
})

players

In [ ]:
from category_encoders.binary import BinaryEncoder
# create a Binaryencoder object
be = BinaryEncoder(cols = ['Club'])
# fit and transform on the data
players_encoded = be.fit_transform(players)

players_encoded

Another example. Note: the dataset [fifa19.csv](https://www.kaggle.com/datasets/flop71/fifa19csv) is found in Kaggle.

In [ ]:
players = pd.read_csv("./data/week_2_fifa19.csv")

players.head()

In [ ]:
clubs = players['Club'].unique()

print(f"There are {len(clubs)} unique clubs in the dataset.")

In [ ]:
# create a Binaryencoder object
be = BinaryEncoder(cols = ['Club'])
# fit and transform on the data
clubs_encoded = be.fit_transform(players)

binary_cols = [
    f"Club_{i}" for i in range(20)
    if f"Club_{i}" in clubs_encoded.columns
]

comparison = pd.concat(
    [players[["Club"]], clubs_encoded[binary_cols]],
    axis=1,
)

with pd.option_context("display.max_columns", None):
    display(comparison.head(10))

There are **652 unique clubs** in the dataset. Binary encoding requires **10 bits** to represent them because:

- 9 bits provide \(2^9 = 512\) combinations, which is insufficient.
- 10 bits provide \(2^{10} = 1024\) combinations, which is sufficient.

Therefore, the encoded columns run from `Club_0` to `Club_9`, giving **10 columns in total**.

An 11-bit representation (as shown in the [kdnuggets article](https://www.kdnuggets.com/2019/06/hitchhikers-guide-feature-extraction.html) would also work, but it would use one more bit than necessary.

### HashingEncoder

One can think of Hashing Encoder as a black box function that converts a string to a number between 0 to some prespecified value.

It differs from binary encoding as in binary encoding two or more of the club parameters could have been 1 while in hashing only one value is 1.

We can use hashing as:

In [ ]:
players = pd.read_csv("./data/week_2_fifa19.csv")

from category_encoders.hashing import HashingEncoder
# create a HashingEncoder object
he = HashingEncoder(cols = ['Club'])
# fit and transform on the data
clubs_encoded = he.fit_transform(players)

binary_cols = [
    f"Club_{i}" for i in range(20)
    if f"Club_{i}" in clubs_encoded.columns
]

clubs_encoded = clubs_encoded.filter(regex=r"^col_\d+$")
comparison = pd.concat(
    [players[["Club"]], clubs_encoded],
    axis=1,
)

with pd.option_context("display.max_columns", None):
    display(comparison.head(5))

There are bound to be collisions (two clubs having the same encoding. For example, Juventus and Paris Saint-Germain (PSG) have the same encoding) but sometimes this technique works well.

That’s a **hash collision**: `HashingEncoder` maps both club names to the same output column.

With `n_components=8`, there are only eight possible buckets for 652 clubs, so many clubs must share an encoding. Hashing does **not** guarantee a unique representation for each club.

You can reduce collisions by increasing the number of columns: `he = HashingEncoder(cols=["Club"], n_components=32)`

As shown in the [reference documentation for `Hashing`](https://contrib.scikit-learn.org/category_encoders/hashing.html) the `n_components`parameter by default is 8, which means that the output will have 8 columns.

### Target/Mean Encoding

This is a technique that I found works pretty well in Kaggle competitions. If both training/test comes from the same dataset from the same time period(cross-sectional), we can get crafty with features.

For example: In the Titanic knowledge challenge, the test data is randomly sampled from the train data. In this case, we can use the target variable averaged over different categorical variable as a feature.

In Titanic, we can create a target encoded feature over the PassengerClass variable.

We have to be careful when using Target encoding as it might induce overfitting in our models. Thus we use k-fold target encoding when we use it.

Note: I think the dataset is found in [Titanic dataset](https://www.kaggle.com/c/titanic)

We can then create a mean encoded feature as:

In [ ]:
import pandas as pd
from sklearn.preprocessing import TargetEncoder

train = pd.read_csv("./data/titanic/train.csv")

targetc = TargetEncoder(
    target_type="binary",
    smooth=0,
    cv=5,
    shuffle=True,
    random_state=42,
)

new_train = train.copy()
new_train["Pclass_Kfold_Target_Enc"] = targetc.fit_transform(
    train[["Pclass"]],
    train["Survived"],
).ravel()

new_train[["Pclass_Kfold_Target_Enc", "Pclass"]].head(10)

In [ ]:
new_train.loc[
    new_train["Pclass"] == 3,
    "Pclass_Kfold_Target_Enc",
].value_counts()

The article states:
*You can see how the passenger class 3 gets encoded as 0.261538 and 0.230570 based on which fold the average is taken from.* But I can see this does not match. **The encoded averages depend on which passengers are assigned to each fold.** Our replacement uses shuffled folds with `random_state=42`, so it needn’t reproduce the tutorial’s values.

This feature is pretty helpful as it encodes the value of the target for the category. Just looking at this feature, we can say that the Passenger in class 1 has a high propensity of surviving compared with Class 3.

## 4. Using Latitude and Longitude features:

This part will tread upon how to use Latitude and Longitude features well.

For this task, I will be using Data from the Playground competition: [New York City Taxi Trip Duration]

To unzip the data:

```bash
unzip data/nyc-taxi-trip-duration.zip train.zip -d data/
unzip data/train.zip -d data/nyc-taxi-train/
rm data/train.zip
```

The train data looks like:

In [ ]:
taxi_train = pd.read_csv("./data/nyc-taxi-train/train.csv")

In [ ]:
taxi_train[['id', 'vendor_id', 'pickup_datetime', 'dropoff_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'store_and_fwd_flag', 'trip_duration']].head()

### A. Haversine Distance Between the Two Lat/Lons:

The haversine formula determines the great-circle distance between two points on a sphere given their longitudes and latitudes



In [ ]:
import numpy as np

def haversine_array(lat1, lng1, lat2, lng2):
    lat1, lng1, lat2, lng2 = map(np.radians, (lat1, lng1, lat2, lng2))
    AVG_EARTH_RADIUS = 6371 # in km
    lat = lat2 - lat1
    lng = lng2 - lng1
    d = np.sin(lat * 0.5) ** 2 + np.cos(lat1) * np.cos(lat2) *      np.sin(lng * 0.5) ** 2
    h = 2 * AVG_EARTH_RADIUS * np.arcsin(np.sqrt(d))
    return h

We could then use the function as:

In [ ]:
taxi_train["haversine_distance"] = taxi_train.apply(
    lambda x: haversine_array(
        x["pickup_latitude"],
        x["pickup_longitude"],
        x["dropoff_latitude"],
        x["dropoff_longitude"],
    ),
    axis=1,
)


To summarize the distances (in kilometers except for count) we can use the `describe()` method, which provides a statistical summary of the data, including count, mean, standard deviation, minimum, maximum, and quartiles.:

In [ ]:
taxi_train["haversine_distance"].describe()

In [ ]:
summary = taxi_train["haversine_distance"].describe().to_frame(name="Value")

summary["Unit"] = ["trips"] + ["km"] * 7
summary["Meaning"] = [
    "Trips with a non-missing distance",
    "Average distance",
    "Standard deviation",
    "Smallest distance",
    "25% of trips are at or below this distance",
    "Median distance",
    "75% of trips are at or below this distance",
    "Largest distance",
]

summary.index.name = "Statistic"

display(summary.style.format({"Value": "{:,.2f}"}))

In [ ]:
taxi_train["haversine_distance"].describe().apply(lambda x: f"{x:,.2f}")

### B. Manhattan Distance Between the two Lat/Lons:

The distance between two points measured along axes at right angles



In [ ]:
def dummy_manhattan_distance(lat1, lng1, lat2, lng2):
    a = haversine_array(lat1, lng1, lat1, lng2)
    b = haversine_array(lat1, lng1, lat2, lng1)
    return a + b

In [ ]:
taxi_train['manhattan_distance'] = taxi_train.apply(
    lambda x: dummy_manhattan_distance(
        x['pickup_latitude'],
        x['pickup_longitude'],
        x['dropoff_latitude'],
        x['dropoff_longitude']
    ),
    axis=1
)

In [ ]:
taxi_train["manhattan_distance"].describe().apply(lambda x: f"{x:,.2f}")

### C. Bearing Between the two Lat/Lons:

A bearing is used to represent the direction of one point relative to another point.

In [ ]:
def bearing_array(lat1, lng1, lat2, lng2):
    AVG_EARTH_RADIUS = 6371 # in km
    lng_delta_rad = np.radians(lng2 - lng1)
    lat1, lng1, lat2, lng2 = map(np.radians, (lat1, lng1, lat2, lng2))
    y = np.sin(lng_delta_rad) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(lng_delta_rad)
    return np.degrees(np.arctan2(y, x))

In [ ]:
taxi_train['bearing'] = taxi_train.apply(
    lambda x: bearing_array(
        x['pickup_latitude'],
        x['pickup_longitude'],
        x['dropoff_latitude'],
        x['dropoff_longitude']
    ),
    axis=1
)


In [ ]:
taxi_train["bearing"].describe()


Bearing is measured in **degrees**:

- **0°** = north
- **90°** = east
- **−90°** = west
- **±180°** = south

In [ ]:
summary = taxi_train["bearing"].describe().to_frame(name="Value")

summary["Unit"] = ["trips"] + ["degrees"] * 7
summary["Meaning"] = [
    "Trips with a non-missing bearing",
    "Arithmetic mean of bearings",
    "Standard deviation of numeric bearings",
    "Smallest numeric bearing",
    "25th percentile",
    "Median numeric bearing",
    "75th percentile",
    "Largest numeric bearing",
]

summary.index.name = "Statistic"

display(summary.style.format({"Value": "{:,.2f}"}))

### D. Center Latitude and Longitude between Pickup and Dropoff:

In [ ]:
taxi_train.loc[:, 'center_latitude'] = (taxi_train['pickup_latitude'].values + taxi_train['dropoff_latitude'].values) / 2
taxi_train.loc[:, 'center_longitude'] = (taxi_train['pickup_longitude'].values + taxi_train['dropoff_longitude'].values) / 2

In [ ]:
taxi_train[['trip_duration', 'haversine_distance', 'manhattan_distance', 'bearing', 'center_latitude', 'center_longitude']].head()

## 6. Some Normal Things you can do with your features:


In [ ]:
import plotly.express as px

plot_data = taxi_train.loc[
    taxi_train["trip_duration"].between(0, 5200)
]

fig = px.histogram(
    plot_data,
    x="trip_duration",
    title="Trip durations up to 5,200 seconds",
    labels={"trip_duration": "Trip duration (seconds)"},
)

fig.update_traces(xbins=dict(start=0, end=5200, size=200))
fig.show()

And with log transformation:

In [ ]:
#taxi_train['log_trip_duration'] = taxi_train['trip_duration'].apply(lambda x: np.log(1+x))

taxi_train["log_trip_duration"] = np.log1p(taxi_train["trip_duration"])

fig = px.histogram(
    taxi_train,
    x="log_trip_duration",
    nbins=80,
    title="Distribution of Log-Transformed Trip Duration",
    labels={"log_trip_duration": "ln(1 + trip duration in seconds)"},
)

fig.show()

A log transformation on trip duration is much less skewed and thus much more helpful for a model.